# 06 Stage-2 Phase 1c — retrain on REAL detector boxes + background class

**Training + evaluation. This is Phase 1c of the two-stage detect-then-refine plan
(`docs/small_object_research_notes.md`).** The submission model is still V6 (~0.234); this
notebook tests whether a Stage-2 refiner trained on the *real* Stage-1 (V6) boxes — with a
**background class** to reject false positives — beats V6 on the comparable full-image Mask mAP.

## Why this run exists (what Phase 1a/1b showed)

- **Phase 1a (gate): PASSED.** V6 at **conf=0.05** localizes most small Caries with support
  (recall@IoU0.3: Caries 1/2/3/5 ~= 0.89/0.73/0.58/0.80). The boxes exist.
- **Phase 1b (transfer): WEAK.** Feeding V6 boxes into the Phase-0 model (trained only on tight GT
  boxes, **no background class**) collapsed the oracle's Caries gains back to ~V6, and the realistic
  `full@0.05` pipeline was **below** V6 (no way to reject FP boxes). The pre-registered rule was
  not cleanly met, but the failure is attributable to two fixable things this notebook addresses:
  the **GT->V6 box-framing domain gap** and the **missing background class**.

## What Phase 1c changes vs Phase 0 (`src/04`)

1. **Training boxes come from V6**, not GT. Run V6 on the TRAIN split at **conf=0.05** and crop ROIs
   around V6's predicted boxes (so Stage 2 learns V6's box statistics, not GT's tight framing).
2. **Background class added** (`nc+1`, index = `num_classes`, empty-mask target). V6 boxes that match
   no GT become hard-negative background samples so Stage 2 can reject FPs at inference.
3. **Warm-started** from `stage2_best.pt` (encoder/decoder/seg head copied; the classifier head is
   re-grown nc -> nc+1 and the overlapping rows copied).

## Decisions baked in (set with the user)

- **Stage-1 conf = 0.05** for both training-box generation and inference (lower conf keeps small-Caries
  recall; 0.25 would discard 40-60% of Caries 2/3/5).
- **Foreground match = box-IoU >= 0.5** to a GT (mAP50-95 rewards tight localization). Boxes with IoU
  in **[0.3, 0.5) are IGNORED** (ambiguous — neither clean FG nor clean background), IoU < 0.3 ->
  background.
- **Background subsampled to ~3:1** (background:foreground) so the classifier is not swamped.

## Pre-registered reading rules (unchanged from the research note)

- Decision metric = comparable full-image Mask mAP / per-class AP (the §8 evaluator, identical to
  `src/03`/`src/04`), **not** ROI-level accuracy.
- Judge only small Caries with support (1/2/5; Caries 4 n=4 / Caries 6 n=5 are noise).
- Large classes (Abrasion/Crown/Filling) route to V6 — the **hybrid** number reflects that.
- `full@0.05` must now be the headline (background class = real FP rejection). Compare to V6 = 0.2099.


## 1. Setup

In [ ]:
# Ultralytics (Stage 1 = V6) + smp (Stage 2). Training kernels usually have Internet on.
try:
    import ultralytics, segmentation_models_pytorch  # noqa
    print("ultralytics + smp already present")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "ultralytics", "segmentation-models-pytorch"])

In [ ]:
import os, json, math, random, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import cv2
import yaml
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import segmentation_models_pytorch as smp
from ultralytics import YOLO

cv2.setNumThreads(0)          # avoid cv2-thread contention in DataLoader workers
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| smp", smp.__version__, "| device:", DEVICE)

## 2. Configuration

`ROI_INPUT` / `PAD_MODE` / `PAD_FACTOR` are kept identical to the Phase-0 `stage2_best.pt` run so the
warm start is meaningful. The Phase-1c-specific knobs (Stage-1 conf, FG/ignore IoU bands, background
ratio) are grouped below.

In [ ]:
# --- ROI framing (MUST match the stage2_best.pt warm-start run) ---
ROI_INPUT   = 224
PAD_MODE    = "relative"
PAD_FACTOR  = 1.5
PAD_ABS_PX  = 48
ENCODER     = "resnet18"

# --- Stage 1 (V6) detector ---
IMG_SIZE_DET = 768            # V6 trained at 768
CAPTURE_CONF = 0.01           # run V6 once at very low conf; filter to STAGE1_CONF in python
DET_IOU      = 0.60           # YOLO NMS IoU
MAX_DET      = 300
STAGE1_CONF  = 0.05           # operating conf (training-box gen + inference) — set with user

# --- Phase 1c label assignment (V6 box -> GT) ---
FG_IOU       = 0.50           # box-IoU >= this to a GT  -> foreground (that GT's class + mask)
IGNORE_IOU   = 0.30           # IoU in [IGNORE_IOU, FG_IOU) -> dropped (ambiguous)
                              # IoU < IGNORE_IOU            -> background
BG_PER_FG    = 3.0            # subsample background to ~ BG_PER_FG : 1 vs foreground

# --- Stage-2 training ---
ENCODER_WEIGHTS = None        # we warm-start from stage2_best.pt, not ImageNet
EPOCHS       = 30
BATCH_SIZE   = 32
LR           = 1e-3
NUM_WORKERS  = 4

# --- Eval (comparable full-image Mask mAP, same as src/03/04) ---
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)
PIPE_CONFS   = [0.05, 0.25]   # score the full pipeline at these Stage-1 confs

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# V6 per-class Mask mAP50-95 reference (src/03 same-code run) + Phase-0 oracle context.
V6_REF_AP = {"abrasion":0.6471,"crown":0.6313,"filling":0.2799,"caries 1":0.1195,
             "caries 2":0.0845,"caries 3":0.0116,"caries 4":0.0000,"caries 5":0.1097,"caries 6":0.0051}
V6_REF_MAP = 0.2099
ORACLE_REF_MAP = 0.312
# Large/common classes routed to V6 in the hybrid (the V13 lesson); the rest (Caries) use Stage 2.
LARGE_CLASSES = {"abrasion", "crown", "filling"}

def _norm_class_key(nm):
    s = str(nm).lower().replace("class", "")
    return " ".join(s.split())

print("ROI", ROI_INPUT, PAD_MODE, PAD_FACTOR, "| stage1 conf", STAGE1_CONF,
      "| FG/ignore IoU", FG_IOU, IGNORE_IOU, "| bg:fg", BG_PER_FG)

## 3. Locate dataset + collect GT (train & val)

Same auto-detection as `src/04`. We need GT polygons for **both** splits: train GTs label V6's train
boxes (FG/BG); val GTs are the eval ground truth.

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yc = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))
if not yc:
    raise FileNotFoundError("No yolo_seg_train.yaml under /kaggle/input.")
DATA_YAML = yc[0]
dcfg = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
names = dcfg.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
num_classes = len(names)
BG_IDX = num_classes                      # background class index (nc+1 total outputs)
n_out  = num_classes + 1
dataset_root = DATA_YAML.parent
yaml_root = Path(dcfg["path"]) if dcfg.get("path") else dataset_root
if not yaml_root.is_absolute():
    yaml_root = dataset_root / yaml_root

def resolve_split(v):
    if v is None: return None
    p = Path(v)
    if p.is_absolute(): return p
    for c in (yaml_root / p, dataset_root / p):
        if c.exists(): return c
    return yaml_root / p

train_images = resolve_split(dcfg.get("train"))
val_images   = resolve_split(dcfg.get("val", dcfg.get("valid")))

def labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images"); parts[i] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
def parse_seg_line(line):
    parts = line.strip().split()
    if len(parts) < 7: return None
    try:
        cls = int(float(parts[0])); coords = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if len(coords) % 2: coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    return (cls, poly) if len(poly) >= 3 else None

def collect_gt(images_dir):
    lbl = labels_dir_for(images_dir)
    out = {}
    for ip in sorted(p for p in Path(images_dir).iterdir() if p.suffix.lower() in IMG_EXTS):
        objs = []
        lp = lbl / (ip.stem + ".txt")
        if lp.exists():
            for line in lp.read_text().splitlines():
                pr = parse_seg_line(line)
                if pr is not None: objs.append(pr)
        out[str(ip)] = objs
    return out

train_gt = collect_gt(train_images)
val_gt   = collect_gt(val_images)
n_gt = np.zeros(num_classes, dtype=int)
for objs in val_gt.values():
    for c, _ in objs: n_gt[c] += 1
print("classes:", names, "| BG_IDX:", BG_IDX)
print("train images:", len(train_gt), "| val images:", len(val_gt))
print("val per-class GT:", {names[c]: int(n_gt[c]) for c in range(num_classes)})

## 4. Load Stage 1 (V6) + warm-start Stage 2 from `stage2_best.pt` (nc -> nc+1)

In [ ]:
MANUAL_V6_PATH = None     # set to exact /kaggle/input path if auto-detect misfires
MANUAL_S2_PATH = None

def find_pt(include, exclude=()):
    cands = []
    for p in Path("/kaggle/input").rglob("*.pt"):
        t = str(p).lower()
        if any(x in t for x in exclude): continue
        score = sum(10 for k in include if k in t) + (20 if p.name.lower().endswith("best.pt") else 0)
        if score > 0: cands.append((score, p))
    cands.sort(key=lambda z: z[0], reverse=True)
    return [p for _, p in cands]

v6_cands = find_pt(["version6", "v6", "yolov8s", "img768"], exclude=["stage2"])
s2_cands = find_pt(["stage2"])
V6_PATH = Path(MANUAL_V6_PATH) if MANUAL_V6_PATH else (v6_cands[0] if v6_cands else None)
S2_PATH = Path(MANUAL_S2_PATH) if MANUAL_S2_PATH else (s2_cands[0] if s2_cands else None)
if V6_PATH is None or not V6_PATH.exists():
    raise FileNotFoundError("No V6 detector .pt found. Add it as Kaggle input or set MANUAL_V6_PATH.")
if S2_PATH is None or not S2_PATH.exists():
    raise FileNotFoundError("No stage2_best.pt found. Add it as Kaggle input or set MANUAL_S2_PATH.")
print("Using V6 :", V6_PATH)
print("Using S2 :", S2_PATH)

detector = YOLO(str(V6_PATH))

# New Stage-2 with nc+1 classification outputs (extra background slot).
model = smp.Unet(encoder_name=ENCODER, encoder_weights=ENCODER_WEIGHTS, in_channels=3,
                 classes=1, activation=None,
                 aux_params=dict(classes=n_out, dropout=0.2)).to(DEVICE)

# Warm start: copy every checkpoint tensor whose shape matches; for the classifier head whose
# out-dim grew (num_classes -> n_out), copy the overlapping rows and leave the background row at init.
ckpt = torch.load(str(S2_PATH), map_location=DEVICE)
ckpt = ckpt.get("state_dict", ckpt) if isinstance(ckpt, dict) else ckpt
tgt = model.state_dict()
copied = grown = skipped = 0
for k, v in ckpt.items():
    if k not in tgt:
        skipped += 1; continue
    if tgt[k].shape == v.shape:
        tgt[k].copy_(v); copied += 1
    elif v.dim() >= 1 and tgt[k].shape[1:] == v.shape[1:] and tgt[k].shape[0] == v.shape[0] + 1:
        tgt[k][:v.shape[0]].copy_(v); grown += 1     # classifier weight/bias: nc -> nc+1
        print(f"  grew {k}: {tuple(v.shape)} -> {tuple(tgt[k].shape)} (bg row left at init)")
    else:
        skipped += 1
model.load_state_dict(tgt)
print(f"warm start: copied {copied}, grown {grown}, skipped {skipped} tensors")

## 5. Run V6 on TRAIN + VAL once (capture low-conf boxes)

One detector pass per image at `CAPTURE_CONF`; boxes stored so any conf >= CAPTURE_CONF is a python
filter. Boxes are used **class-agnostically** for cropping (Stage 2 re-classifies).

In [ ]:
def run_detector(gt_dict, tag):
    det, wh = {}, {}
    for i, ip in enumerate(gt_dict):
        img = cv2.imread(ip, cv2.IMREAD_COLOR)
        h, w = img.shape[:2]; wh[ip] = (w, h)
        res = detector.predict(source=img, imgsz=IMG_SIZE_DET, conf=CAPTURE_CONF, iou=DET_IOU,
                               max_det=MAX_DET, device=DEVICE, task="segment",
                               verbose=False, save=False)[0]
        dets = []
        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy(); conf = res.boxes.conf.cpu().numpy()
            for b, cf in zip(xyxy, conf):
                x0, y0, x1, y1 = [int(round(v)) for v in b]
                x0 = max(0, min(x0, w-1)); y0 = max(0, min(y0, h-1))
                x1 = max(x0+1, min(x1, w)); y1 = max(y0+1, min(y1, h))
                dets.append({"box": (x0, y0, x1, y1), "conf": float(cf)})
        det[ip] = dets
        if (i+1) % 100 == 0: print(f"  [{tag}] {i+1}/{len(gt_dict)} images")
    print(f"[{tag}] captured {sum(len(d) for d in det.values())} boxes over {len(det)} images")
    return det, wh

det_train, wh_train = run_detector(train_gt, "train")
det_val,   wh_val   = run_detector(val_gt,   "val")

## 6. Label V6 train boxes (FG / IGNORE / background) + subsample background

Each V6 box (conf >= `STAGE1_CONF`) is matched to the best-IoU train GT:
`>= FG_IOU` -> foreground (target = that GT's class + GT polygon, rasterized in the box ROI frame);
`< IGNORE_IOU` -> background (empty mask, class `BG_IDX`); the `[IGNORE_IOU, FG_IOU)` band is dropped.
Background is then subsampled to ~`BG_PER_FG : 1`.

In [ ]:
def gt_pixel_bbox(poly_norm, w, h):
    xs = poly_norm[:,0]*w; ys = poly_norm[:,1]*h
    return (max(0,int(np.floor(xs.min()))), max(0,int(np.floor(ys.min()))),
            min(w,int(np.ceil(xs.max()))),  min(h,int(np.ceil(ys.max()))))

def box_iou(a, b):
    ix0,iy0 = max(a[0],b[0]), max(a[1],b[1]); ix1,iy1 = min(a[2],b[2]), min(a[3],b[3])
    iw,ih = max(0,ix1-ix0), max(0,iy1-iy0); inter = iw*ih
    if inter <= 0: return 0.0
    aa=(a[2]-a[0])*(a[3]-a[1]); bb=(b[2]-b[0])*(b[3]-b[1])
    return inter/(aa+bb-inter)

fg_samples, bg_samples = [], []
for ip, dets in det_train.items():
    w, h = wh_train[ip]
    gobjs = train_gt[ip]
    gboxes = [(c, gt_pixel_bbox(p, w, h), p) for c, p in gobjs]
    for d in dets:
        if d["conf"] < STAGE1_CONF: continue
        best, bg = 0.0, None
        for (gc, gbx, gpoly) in gboxes:
            v = box_iou(d["box"], gbx)
            if v > best: best, bg = v, (gc, gpoly)
        if best >= FG_IOU:
            fg_samples.append({"image": ip, "box": d["box"], "cls": bg[0],
                               "poly": bg[1], "w": w, "h": h})
        elif best < IGNORE_IOU:
            bg_samples.append({"image": ip, "box": d["box"], "cls": BG_IDX,
                               "poly": None, "w": w, "h": h})
        # else: ignore band -> dropped

random.shuffle(bg_samples)
keep_bg = int(round(BG_PER_FG * len(fg_samples)))
bg_kept = bg_samples[:keep_bg]
train_samples = fg_samples + bg_kept
random.shuffle(train_samples)
print(f"FG boxes: {len(fg_samples)} | BG available: {len(bg_samples)} | BG kept: {len(bg_kept)} "
      f"({BG_PER_FG}:1) | total train ROIs: {len(train_samples)}")
fg_by_cls = pd.Series([s['cls'] for s in fg_samples]).value_counts().sort_index()
print("FG per-class:", {names[i]: int(fg_by_cls.get(i,0)) for i in range(num_classes)})

## 7. ROI crop helpers + on-disk cache + Dataset

Crop ROIs around the **V6 box** (not the GT box) — this is what closes the Phase-1b framing gap.
`crop_roi_from_box` pads the V6 box to a square, crops/resizes, and rasterizes the matched GT polygon
into the ROI frame (empty mask for background). Same on-disk cache trick as `src/04` (each panoramic
decoded once).

In [ ]:
def padded_square_box(x0, y0, x1, y1, w, h):
    bw, bh = x1-x0, y1-y0; cx, cy = (x0+x1)/2.0, (y0+y1)/2.0
    half = (max(bw,bh)*(1.0+PAD_FACTOR)/2.0) if PAD_MODE=="relative" else (max(bw,bh)/2.0 + PAD_ABS_PX)
    half = max(half, 1.0)
    sx0,sy0,sx1,sy1 = cx-half, cy-half, cx+half, cy+half
    sx0 = max(0.0, min(sx0, w-1)); sy0 = max(0.0, min(sy0, h-1))
    sx1 = max(sx0+1, min(sx1, w)); sy1 = max(sy0+1, min(sy1, h))
    return int(round(sx0)), int(round(sy0)), int(round(sx1)), int(round(sy1))

def crop_roi_from_box(img, box_px, poly_norm):
    # Crop ROI around a given pixel box (padded square); rasterize poly_norm into ROI frame.
    h, w = img.shape[:2]
    x0, y0, x1, y1 = padded_square_box(*box_px, w, h)
    roi = cv2.resize(img[y0:y1, x0:x1], (ROI_INPUT, ROI_INPUT), interpolation=cv2.INTER_LINEAR)
    mask = np.zeros((ROI_INPUT, ROI_INPUT), dtype=np.uint8)
    if poly_norm is not None:
        cw, ch = x1-x0, y1-y0
        pts = poly_norm.copy()
        pts[:,0] = (pts[:,0]*w - x0)/cw*ROI_INPUT
        pts[:,1] = (pts[:,1]*h - y0)/ch*ROI_INPUT
        cv2.fillPoly(mask, [pts.astype(np.int32)], 1)
    return roi, (x0, y0, x1, y1), mask

ROI_CACHE = Path("/kaggle/working/roi_cache_p1c")
def build_cache(samples, split):
    idir = ROI_CACHE/split/"img"; mdir = ROI_CACHE/split/"mask"
    idir.mkdir(parents=True, exist_ok=True); mdir.mkdir(parents=True, exist_ok=True)
    by_img = {}
    for idx, s in enumerate(samples): by_img.setdefault(s["image"], []).append((idx, s))
    manifest = []
    for image_path, items in by_img.items():
        img = cv2.imread(image_path, cv2.IMREAD_COLOR)
        if img is None: print("  WARN unreadable", image_path); continue
        for idx, s in items:
            roi, box, mask = crop_roi_from_box(img, s["box"], s["poly"])
            rf = idir/f"{idx:06d}.png"; mf = mdir/f"{idx:06d}.png"
            cv2.imwrite(str(rf), roi); cv2.imwrite(str(mf), (mask*255).astype(np.uint8))
            manifest.append({"roi": str(rf), "mask": str(mf), "cls": int(s["cls"])})
    print(f"  [{split}] cached {len(manifest)} ROIs from {len(by_img)} images")
    return manifest

if ROI_CACHE.exists(): shutil.rmtree(ROI_CACHE)
print("Building Phase-1c ROI cache (train only; val ROIs are cropped live in eval) ...")
train_manifest = build_cache(train_samples, "train")

class ROIDataset(Dataset):
    def __init__(self, manifest, train=False):
        self.m = manifest; self.train = train
    def __len__(self): return len(self.m)
    def __getitem__(self, i):
        e = self.m[i]
        roi = cv2.imread(e["roi"], cv2.IMREAD_COLOR)
        mask = (cv2.imread(e["mask"], cv2.IMREAD_GRAYSCALE) > 127).astype(np.float32)
        roi = cv2.cvtColor(roi, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        if self.train and random.random() < 0.5:
            roi = roi[:, ::-1, :].copy(); mask = mask[:, ::-1].copy()
        roi = (roi - IMAGENET_MEAN)/IMAGENET_STD
        return torch.from_numpy(roi.transpose(2,0,1)), torch.from_numpy(mask)[None], e["cls"]

# Hold out 10% of train ROIs as a Stage-2 val proxy for checkpoint selection (FG mask-IoU + acc).
random.shuffle(train_manifest)
n_hold = max(1, int(0.1*len(train_manifest)))
s2_val_manifest = train_manifest[:n_hold]
s2_trn_manifest = train_manifest[n_hold:]
trn_ds = ROIDataset(s2_trn_manifest, train=True)
val_ds = ROIDataset(s2_val_manifest, train=False)
loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=(DEVICE=="cuda"))
if NUM_WORKERS > 0: loader_kw.update(persistent_workers=True, prefetch_factor=4)
trn_loader = DataLoader(trn_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True, **loader_kw)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **loader_kw)
print("stage2 train ROIs:", len(trn_ds), "| stage2 val ROIs:", len(val_ds))

## 8. Train Stage-2 (background-aware)

Classification CE over `nc+1` (incl. background) on **all** samples; segmentation loss (Dice+BCE)
only on **foreground** samples (background target is empty by construction — don't let it dominate the
mask head). Checkpoint kept by a combined val proxy = FG mask-IoU + overall class accuracy. ROI-level
numbers are diagnostic; the decision metric is the §9 full-image pipeline mAP.

In [ ]:
dice_loss = smp.losses.DiceLoss(mode="binary", from_logits=True)
bce_loss  = nn.BCEWithLogitsLoss()
ce_loss   = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

@torch.no_grad()
def val_proxy():
    model.eval()
    correct = total = 0; iou_sum = fg_n = 0.0
    for roi, mask, cls in val_loader:
        roi = roi.to(DEVICE); mask = mask.to(DEVICE); cls_t = torch.as_tensor(cls).to(DEVICE)
        mlog, clog = model(roi)
        pred = clog.argmax(1)
        correct += (pred == cls_t).sum().item(); total += len(cls_t)
        fg = cls_t != BG_IDX
        if fg.any():
            pm = (torch.sigmoid(mlog[fg]) > 0.5).float()
            inter = (pm*mask[fg]).sum((1,2,3))
            union = ((pm+mask[fg])>0).float().sum((1,2,3)).clamp(min=1)
            iou_sum += (inter/union).sum().item(); fg_n += int(fg.sum())
    acc = correct/max(total,1); miou = iou_sum/max(fg_n,1)
    return acc, miou

best_score, n_batches = -1.0, len(trn_loader)
history = []
for epoch in range(1, EPOCHS+1):
    model.train(); run = rseg = rcls = 0.0
    for roi, mask, cls in trn_loader:
        roi = roi.to(DEVICE); mask = mask.to(DEVICE); cls_t = torch.as_tensor(cls).to(DEVICE)
        optimizer.zero_grad()
        mlog, clog = model(roi)
        lcls = ce_loss(clog, cls_t)
        fg = cls_t != BG_IDX
        lseg = (dice_loss(mlog[fg], mask[fg]) + bce_loss(mlog[fg], mask[fg])) if fg.any() else torch.zeros((), device=DEVICE)
        loss = lseg + lcls
        loss.backward(); optimizer.step()
        run += loss.item(); rseg += float(lseg); rcls += lcls.item()
    acc, miou = val_proxy()
    score = miou + acc      # combined proxy (mask quality + background-aware classification)
    history.append({"epoch": epoch, "train_loss": run/n_batches, "seg": rseg/n_batches,
                    "cls": rcls/n_batches, "val_acc": acc, "val_fg_mask_iou": miou})
    flag = ""
    if score > best_score:
        best_score = score; torch.save(model.state_dict(), "/kaggle/working/stage2_p1c_best.pt"); flag = " *best"
    print(f"ep {epoch:>2}/{EPOCHS} | loss {run/n_batches:.3f} (seg {rseg/n_batches:.3f} cls {rcls/n_batches:.3f}) "
          f"| val acc {acc:.3f} fgIoU {miou:.3f}{flag}")

pd.DataFrame(history).to_csv("/kaggle/working/stage2_p1c_history.csv", index=False)
model.load_state_dict(torch.load("/kaggle/working/stage2_p1c_best.pt"))
model.eval()
print("loaded best (combined proxy =", round(best_score,4), ")")

## 9. Comparable full-image Mask mAP — full V6 -> Stage2 pipeline (the decision metric)

Run V6 (conf >= PIPE_CONFS) on the val images, refine every box through Stage 2, **drop boxes whose
argmax is background**, place masks back to full-image coords, and score with the same mask-mAP code
as `src/03`/`src/04`. Detection score = V6 conf x Stage-2 (foreground) class prob. `full@0.05` is the
headline now that FP rejection exists; `TPonly@0.05` (perfect FP filter) is kept as the refinement
ceiling.

In [ ]:
def gt_local_mask(poly_norm, w, h):
    pts = poly_norm.copy(); pts[:,0]*=w; pts[:,1]*=h
    gx0=max(0,int(np.floor(pts[:,0].min()))); gy0=max(0,int(np.floor(pts[:,1].min())))
    gx1=min(w,int(np.ceil(pts[:,0].max())));  gy1=min(h,int(np.ceil(pts[:,1].max())))
    gw=max(1,gx1-gx0); gh=max(1,gy1-gy0)
    m=np.zeros((gh,gw),dtype=np.uint8); pts[:,0]-=gx0; pts[:,1]-=gy0
    cv2.fillPoly(m,[pts.astype(np.int32)],1)
    return (gx0,gy0,gx1,gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    pa=int(pmask.sum()); ga=int(gmask.sum())
    if pa==0 or ga==0: return 0.0
    ix0=max(pbox[0],gbox[0]); iy0=max(pbox[1],gbox[1]); ix1=min(pbox[2],gbox[2]); iy1=min(pbox[3],gbox[3])
    inter=0
    if ix1>ix0 and iy1>iy0:
        pc=pmask[iy0-pbox[1]:iy1-pbox[1], ix0-pbox[0]:ix1-pbox[0]]
        gc=gmask[iy0-gbox[1]:iy1-gbox[1], ix0-gbox[0]:ix1-gbox[0]]
        inter=int(np.logical_and(pc,gc).sum())
    union=pa+ga-inter
    return inter/union if union>0 else 0.0

@torch.no_grad()
def stage2_on_boxes(img, boxes):
    if not boxes: return []
    h, w = img.shape[:2]; xs=[]; pboxes=[]
    for bx in boxes:
        sb = padded_square_box(*bx, w, h); pboxes.append(sb)
        crop = cv2.resize(img[sb[1]:sb[3], sb[0]:sb[2]], (ROI_INPUT, ROI_INPUT))
        x = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32)/255.0
        xs.append(((x-IMAGENET_MEAN)/IMAGENET_STD).transpose(2,0,1))
    mlog, clog = model(torch.from_numpy(np.stack(xs)).to(DEVICE))
    prob = torch.softmax(clog,1).cpu().numpy()
    masks = (torch.sigmoid(mlog)[:,0] > 0.5).cpu().numpy().astype(np.uint8)
    out=[]
    for i, sb in enumerate(pboxes):
        pm = cv2.resize(masks[i], (sb[2]-sb[0], sb[3]-sb[1]), interpolation=cv2.INTER_NEAREST)
        out.append((sb, prob[i], pm))
    return out

# Cache Stage-2 outputs for ALL captured val boxes once.
val_cache = {}
for ip, dets in det_val.items():
    img = cv2.imread(ip, cv2.IMREAD_COLOR); w, h = wh_val[ip]
    gt_boxes = [gt_pixel_bbox(p, w, h) for _, p in val_gt[ip]]
    s2 = stage2_on_boxes(img, [d["box"] for d in dets])
    items=[]
    for d, (pbox, prob, pm) in zip(dets, s2):
        fg_cls = int(prob[:num_classes].argmax())      # best foreground class
        items.append({"pbox": pbox, "prob": prob, "fg_cls": fg_cls,
                      "is_bg": int(prob.argmax()) == BG_IDX, "pmask": pm, "v6_conf": d["conf"],
                      "matches_gt": any(box_iou(d["box"], gb) >= 0.5 for gb in gt_boxes)})
    val_cache[ip] = items
print("cached stage2 for", sum(len(v) for v in val_cache.values()), "val boxes")

def ap_101(confs, tps, npos):
    if npos==0: return float("nan")
    if not confs: return 0.0
    o=np.argsort(-np.asarray(confs)); tps=np.asarray(tps)[o]
    tp_c=np.cumsum(tps); fp_c=np.cumsum(1-tps); rec=tp_c/npos; prec=tp_c/np.maximum(tp_c+fp_c,1e-9)
    return sum((prec[rec>=r].max() if np.any(rec>=r) else 0.0) for r in np.linspace(0,1,101))/101.0

def pipeline_map(conf_t, tp_only=False, reject_bg=True):
    records={(c,ti):[] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
    for ip, objs in val_gt.items():
        w, h = wh_val[ip]
        gts=[(c, *gt_local_mask(p, w, h)) for c, p in objs]
        preds=[]
        for it in val_cache[ip]:
            if it["v6_conf"] < conf_t: continue
            if tp_only and not it["matches_gt"]: continue
            if reject_bg and it["is_bg"]: continue
            cls = it["fg_cls"]; score = it["v6_conf"]*float(it["prob"][cls])
            preds.append((cls, score, it["pbox"], it["pmask"]))
        for ti, thr in enumerate(IOU_THRESHOLDS):
            order=sorted(range(len(preds)), key=lambda i: preds[i][1], reverse=True)
            used=[False]*len(gts)
            for pi in order:
                pc, sc, pbox, pm = preds[pi]
                best, bg = 0.0, -1
                for gi,(gc,gbox,gm) in enumerate(gts):
                    if used[gi] or gc != pc: continue
                    v=iou_local(pbox,pm,gbox,gm)
                    if v>best: best, bg = v, gi
                tp = 1 if (bg>=0 and best>=thr) else 0
                if tp: used[bg]=True
                records[(pc,ti)].append((sc,tp))
    pac=np.full(num_classes, np.nan)
    for c in range(num_classes):
        if n_gt[c]==0: continue
        pac[c]=np.nanmean([ap_101([r[0] for r in records[(c,ti)]],
                                  [r[1] for r in records[(c,ti)]], n_gt[c])
                           for ti in range(len(IOU_THRESHOLDS))])
    return pac

results={}
for ct in PIPE_CONFS:
    results[f"full@{ct}"] = pipeline_map(ct, tp_only=False, reject_bg=True)
results["TPonly@0.05"] = pipeline_map(0.05, tp_only=True, reject_bg=True)

def hybrid_map(pac):
    vals=[]
    for c in range(num_classes):
        key=_norm_class_key(names[c])
        if key in LARGE_CLASSES:
            v=V6_REF_AP.get(key, np.nan)
        else:
            v=pac[c]
        if not np.isnan(v): vals.append(v)
    return float(np.mean(vals)) if vals else float("nan")

def show(tag, pac):
    valid=~np.isnan(pac)
    print(f"\n=== {tag} === mAP={np.nanmean(pac[valid]):.4f} (V6 {V6_REF_MAP:.4f}, oracle {ORACLE_REF_MAP:.3f}) "
          f"| hybrid(large->V6)={hybrid_map(pac):.4f}")
    for c in range(num_classes):
        v6=V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
        d=pac[c]-v6 if not (np.isnan(pac[c]) or np.isnan(v6)) else float("nan")
        print(f"  {names[c]:>14s} n={int(n_gt[c]):>3d}  pipe={pac[c]:.3f}  V6={v6:.3f}  d={d:+.3f}")
for tag, pac in results.items():
    show(tag, pac)

## 10. Save outputs (Kaggle Save Version)

In [ ]:
rows=[]
for tag, pac in results.items():
    for c in range(num_classes):
        v6=V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
        rows.append({"variant": tag, "class": names[c], "n_gt": int(n_gt[c]),
                     "pipeline_AP": (None if np.isnan(pac[c]) else float(pac[c])),
                     "v6_ref_AP": (None if np.isnan(v6) else float(v6))})
pd.DataFrame(rows).to_csv("/kaggle/working/phase1c_pipeline.csv", index=False)

summary={
    "config": {"ROI_INPUT": ROI_INPUT, "PAD_MODE": PAD_MODE, "PAD_FACTOR": PAD_FACTOR,
               "STAGE1_CONF": STAGE1_CONF, "FG_IOU": FG_IOU, "IGNORE_IOU": IGNORE_IOU,
               "BG_PER_FG": BG_PER_FG, "EPOCHS": EPOCHS, "BATCH_SIZE": BATCH_SIZE, "LR": LR},
    "n_fg_train": len(fg_samples), "n_bg_kept": len(bg_kept),
    "pipeline_mAP": {tag: round(float(np.nanmean(pac[~np.isnan(pac)])),4) for tag,pac in results.items()},
    "hybrid_mAP": {tag: round(hybrid_map(pac),4) for tag,pac in results.items()},
    "v6_ref_mAP": V6_REF_MAP, "oracle_ref_mAP": ORACLE_REF_MAP,
}
with open("/kaggle/working/phase1c_summary.json","w") as f:
    json.dump(summary, f, indent=2)

print("Saved to /kaggle/working:")
for p in ["stage2_p1c_best.pt","stage2_p1c_history.csv","phase1c_pipeline.csv","phase1c_summary.json"]:
    fp=Path("/kaggle/working")/p
    print(f"  [{'OK' if fp.exists() else 'MISSING'}] {p}")
print("\npipeline mAP:", json.dumps(summary["pipeline_mAP"], indent=2))
print("hybrid   mAP:", json.dumps(summary["hybrid_mAP"], indent=2))

## 11. How to read this -> decide Phase 1c verdict

Compare against **V6 = 0.2099** and the Phase-1b numbers (full@0.05 = 0.182, TPonly@0.05 = 0.218):

- **Did the background class fix the FP problem?** `full@0.05` should now be much closer to (or above)
  `TPonly@0.05`. If `full` is still far below `TPonly`, background rejection is the bottleneck (tune
  `BG_PER_FG`, the background-vs-FG loss weighting, or the rejection threshold).
- **Did training on V6 boxes recover the oracle Caries gains?** Look at small Caries (1/2/5) `pipe` vs
  `V6`. Phase 1b TPonly was ~flat (Caries1/2) to slightly up (Caries5 +0.05). If Phase 1c clearly
  lifts these beyond noise, the framing-gap hypothesis is confirmed.
- **Headline decision = `hybrid(large->V6)` mAP vs 0.2099.** Caries are low-weight, so even a clean win
  on them moves the aggregate modestly; the hybrid (large classes kept on V6) is the realistic deploy
  number. If hybrid > V6 beyond the ~0.003 noise band, the two-stage system is worth integrating into
  a submission; if not, Stage 2 stays a research result and V6 remains the submission model.
